# Solutions to project 2: one column at a time, with NumPy

Project 1 looped over the events one at a time.
Here each quantity is a NumPy array with one entry per event, and one line of code handles every event.

<img src="../img/loop-vs-column.svg" width="80%" alt="">

The file holds every event of the full sample that has exactly two electrons and two muons: 30,868 events.

**How to work**

- Run each cell with `Shift` + `Enter`, from top to bottom.
- Replace every `...` with your own code.
- Solutions are in the `solutions` folder. Try for five minutes before you look.

In [ ]:
import hist
import matplotlib.pyplot as plt
import numpy as np

import awkward as ak  # only used here, to read the file into NumPy arrays

raw = ak.from_parquet("../data/SMHiggsToZZTo4L.parquet")
raw = raw[(ak.num(raw.electron) == 2) & (ak.num(raw.muon) == 2)]
fields = ["pt", "eta", "phi", "mass", "charge"]
e1 = {field: ak.to_numpy(raw.electron[field][:, 0]) for field in fields}
e2 = {field: ak.to_numpy(raw.electron[field][:, 1]) for field in fields}
mu1 = {field: ak.to_numpy(raw.muon[field][:, 0]) for field in fields}
mu2 = {field: ak.to_numpy(raw.muon[field][:, 1]) for field in fields}

Each lepton is a dictionary again, as in project 1.
Now every value is an array with one entry per event.
`{field: ... for field in fields}` builds the dictionary in one line, in the same way as a list comprehension.

<img src="../img/lepton-columns.svg" width="75%" alt="">

In [ ]:
e1["pt"], e1["pt"].shape

## Part 1: Masks

A comparison on a whole array gives a mask.

**Tools.** `e1["pt"] > 20` gives one `True` or `False` per event.
Combine masks with `&` (and) and `|` (or), and put each condition in parentheses.
`mask.sum()` counts the `True` values.

**Task.** How many events have both electrons with $p_T$ above 20 GeV?

<img src="../img/mask.svg" width="70%" alt="">

In [ ]:
n_both = ((e1["pt"] > 20) & (e2["pt"] > 20)).sum()
n_both

## Part 2: Functions on arrays

Your functions from project 1 work on whole arrays.

`math.cos` accepts one number. `np.cos` accepts a whole array and returns an array.
Nothing else in your functions needs to change.

**Task.** Copy `four_momentum`, `add` and `mass` from project 1 and replace every `math.` with `np.`.
Then run the cell to get the mass of every electron pair at once.

In [ ]:
def four_momentum(particle):
    px = particle["pt"] * np.cos(particle["phi"])
    py = particle["pt"] * np.sin(particle["phi"])
    pz = particle["pt"] * np.sinh(particle["eta"])
    E = np.sqrt(px**2 + py**2 + pz**2 + particle["mass"] ** 2)
    return {"E": E, "px": px, "py": py, "pz": pz}


def add(p, q):
    return {"E": p["E"] + q["E"], "px": p["px"] + q["px"], "py": p["py"] + q["py"], "pz": p["pz"] + q["pz"]}


def mass(p):
    return np.sqrt(p["E"] ** 2 - p["px"] ** 2 - p["py"] ** 2 - p["pz"] ** 2)


z_ee = add(four_momentum(e1), four_momentum(e2))
mass(z_ee)

## Part 3: The Z mass

One line gives the Z mass in every event.

**Task.** Build `z_mumu` from the two muons in the same way.

In [ ]:
z_mumu = add(four_momentum(mu1), four_momentum(mu2))
mass(z_mumu)

**Plot.** Both distributions have a peak at 91 GeV, from the Z that has its full mass.
The spread below the peak comes from the lighter Z.

In [ ]:
h_ee = hist.Hist.new.Reg(60, 0, 120, label="pair mass [GeV]").Double().fill(mass(z_ee))
h_mumu = hist.Hist.new.Reg(60, 0, 120, label="pair mass [GeV]").Double().fill(mass(z_mumu))
h_ee.plot(label="two electrons")
h_mumu.plot(label="two muons")
plt.legend();

## Part 4: Selecting events

A mask selects events without a loop.

In project 1 you kept the events where both pairs have opposite charge.

**Task.** Make the mask `opposite`: `True` where the two electrons have opposite charge and the two muons have opposite charge.

In [ ]:
opposite = (e1["charge"] != e2["charge"]) & (mu1["charge"] != mu2["charge"])
opposite

## Part 5: The Higgs mass

One line gives the Higgs mass of every event.

**Task.** Compute the Higgs four-momentum of every event, then the masses of the events in `opposite`.
Compare the time with Part 6 of project 1, which went through one sixth as many events.

In [ ]:
%%time
higgs = add(z_ee, z_mumu)
higgs_masses = mass(higgs)[opposite]

In [ ]:
higgs_masses

In [ ]:
h = hist.Hist.new.Reg(50, 80, 180, label="four-lepton mass [GeV]").Double()
h.fill(higgs_masses)
h.plot();

## Part 6: Z mass windows

`np.minimum` picks the smaller value, event by event.

Apply the Z mass cuts from the bonus of project 1: the lighter Z between 12 and 120 GeV, the heavier between 40 and 120 GeV.

**Tools.** `np.minimum(a, b)` and `np.maximum(a, b)` compare two arrays element by element.
`12 < x < 120` does not work on arrays. Write two comparisons joined by `&`.

**Task.** Complete `z_small`, `z_big` and the mask `z_cut`.

In [ ]:
z_small = np.minimum(mass(z_ee), mass(z_mumu))
z_big = np.maximum(mass(z_ee), mass(z_mumu))
z_cut = (12 < z_small) & (z_small < 120) & (40 < z_big) & (z_big < 120)
higgs_masses_cut = mass(higgs)[opposite & z_cut]
higgs_masses_cut

In [ ]:
h_cut = hist.Hist.new.Reg(50, 80, 180, label="four-lepton mass [GeV]").Double()
h_cut.fill(higgs_masses_cut)
h.plot(label="opposite charges")
h_cut.plot(label="and Z mass cuts")
plt.legend();

## Bonus: A 2D histogram

A 2D histogram shows both Z masses at once.

**Task.** Fill the histogram with the two pair masses of the events in `opposite`.
Every event has one pair near 91 GeV, so the counts gather in two bands, one at 91 GeV on each axis.

In [ ]:
h2 = (
    hist.Hist.new
    .Reg(60, 0, 120, name="ee", label="two-electron mass [GeV]")
    .Reg(60, 0, 120, name="mumu", label="two-muon mass [GeV]")
    .Double()
)
h2.fill(ee=mass(z_ee)[opposite], mumu=mass(z_mumu)[opposite])
h2.plot2d();

**Done.** Project 3 wraps this analysis in a class. Project 4 uses every event, whatever its number of leptons.